In [86]:
from pathlib import Path
from datetime import date

import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
from statsmodels.regression import linear_model

In [87]:
collected_vehicle_data_path = Path("..", "raw_data", "collected-data", "collected_vehicle_data.csv")

# Electric Vehicles

## Results

In [88]:
obs = pl.scan_csv(
    source=collected_vehicle_data_path,
    has_header=True
)

In [89]:
n = (obs
    .select(pl.len())  
    .collect() 
    .item()
)

num_elec = (obs
    .filter(pl.col("engine") == "ELECTRIC")
    .select(pl.len())  
    .collect() 
    .item()
)

There were `{python} num_elec` electric vehicles in the sample of `{python} n` vehicles.

## Sensitivity Analysis

In [90]:
elec_regs = pl.LazyFrame(
    data=dict(
        cutoff_date=pl.Series([date(2023, 2, 15), date(2024, 2, 15), date(2025, 2, 17)]),
        p=pl.Series([5601/488967, 8414/507164, 12153/525078])
    )
)
elec_regs.collect()

cutoff_date,p
date,f64
2023-02-15,0.011455
2024-02-15,0.01659
2025-02-17,0.023145


In [91]:
x = elec_regs.select("cutoff_date").collect().to_series()
y = elec_regs.select("p").collect().to_series()
fig = px.line(
    x=x, 
    y=y, 
    markers="lines+markers"
)
fig.update_layout(
    title="Utah County Electric Vehicle Registrations",
    xaxis=dict(title="Cutoff Date"),
    yaxis=dict(title="Relative Frequency")
)

/home/justin/bin/mambaforge/envs/justins_room/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



In [119]:
# x is the time since the minimum time
# that we have registration data for.
x = elec_regs.select(pl.col("cutoff_date").cast(pl.Float64) - pl.col("cutoff_date").min().cast(pl.Float64)).collect().to_series().to_numpy()
y = elec_regs.select("p").collect().to_series().to_numpy()
# Fit a linear model using OLS.
exog = np.reshape(x, (-1, 1))
exog = np.insert(arr=exog, obj=0, values=1, axis=1)
endog = np.log(y)

elec_regs_ols = linear_model.OLS(
    endog=endog,
    exog=exog,
    hasconst=True
)

In [118]:
# exog = np.reshape(exog, (-1, 1))
exog

array([[  1.,   0.],
       [  1., 365.],
       [  1., 733.]])

In [109]:
endog = np.reshape(endog, (-1, 1))
endog

array([[ 1.        ],
       [-4.46934985],
       [-4.09893744],
       [-3.76597077]])

In [125]:
np.linalg.inv(exog.T @ exog) @ exog.T @ endog

array([-4.46260163e+00,  9.59514415e-04])

In [122]:
elec_regs_ols_res = elec_regs_ols.fit()

In [123]:
elec_regs_ols_res.summary()


/home/justin/bin/mambaforge/envs/justins_room/lib/python3.11/site-packages/statsmodels/stats/stattools.py:74: ValueWarning:

omni_normtest is not valid with less than 8 observations; 3 samples were given.



<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.999
Model:                            OLS   Adj. R-squared:                  0.998
Method:                 Least Squares   F-statistic:                     912.6
Date:                Wed, 28 May 2025   Prob (F-statistic):             0.0211
Time:                        20:46:20   Log-Likelihood:                 9.7111
No. Observations:                   3   AIC:                            -15.42
Df Residuals:                       1   BIC:                            -17.23
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -4.4626      0.015   -297.196      0.002      -4.653      -4.272
x1             0.0010   3.18e-05     30.210      0.021       0.001       0.001
==============================================================================
Omnibus:                          nan   Durbin-Watson:                   3.000
Prob(Omnibus):                    nan   Jarque-Bera (JB):                0.531
Skew:                           0.707   Prob(JB):                        0.767
Kurtosis:                       1.500   Cond. No.                         747.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [ ]:
elec_regs_ols_preds = elec_regs_ols_res.predict()
elec_regs_ols_preds_orig = np.exp(elec_regs_ols_preds)
elec_regs_ols_preds_orig

array([0.01153232, 0.01636879, 0.02330057])

In [129]:
x = elec_regs.select("cutoff_date").collect().to_series()
y = elec_regs.select("p").collect().to_series()
fig = px.line(
    x=x, 
    y=y, 
    markers="lines+markers"
)
fig.add_trace(
    go.Scatter(
        name="Regression Estimate",
        x=x,
        y=elec_regs_ols_preds_orig
    )
)
fig.update_layout(
    title="Relative Frequency of Electric Vehicles in the Target Population",
    xaxis=dict(title="Cutoff Date"),
    yaxis=dict(title="Relative Frequency")
)

/home/justin/bin/mambaforge/envs/justins_room/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



Based on the lower number of electric vehicles in the sample than expected, we think it best to extrapolate the registration data itself using a constant function to form our estimate for March 2025.